# 4. Evaluation: how often is the chatbot right?

A RAG demo that answers three questions well proves nothing. This notebook measures it on a **test set** of questions with known answers.

**Two things are measured separately**, because they fail for different reasons:

| Part | Question | Metrics |
|---|---|---|
| **Retrieval** | Did the search find a chunk that contains the answer? | Hit rate@k, MRR |
| **Answer** | Is the final answer correct? Does it refuse questions it can't answer? | Accuracy, refusal rate |

**Groq free tier:** 8,000 tokens per minute (about 3 chatbot answers per minute). All results are **saved to CSV files**,
so re-running the notebook reads them instead of calling Groq again. Delete a file to regenerate it.

## Setup

In [1]:
from pathlib import Path

# Project root = the folder that contains requirements.txt (works no matter where the notebook runs from)
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print("Project folder:", ROOT)

Project folder: /Users/os_bhosale/Desktop/Projects/Financial-Filings-Rag


In [2]:
import sys
import json
import re
import pandas as pd
from langchain_core.prompts import ChatPromptTemplate

sys.path.insert(0, str(ROOT / "3_rag"))
from rag_pipeline import Retriever, load_llm, generate_answer, COMPANIES, METHODS, NOT_FOUND

EVAL_DIR = ROOT / "4_evaluation"
retriever = Retriever()
chunks = retriever.chunks
helper_llm = load_llm("openai/gpt-oss-20b")      # smaller model for writing test questions and grading
answer_llm = load_llm()                           # the chatbot's model (gpt-oss-120b)

W1108 16:24:09.170000 2883 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

## Step 1: Build the test set

**In-scope questions** are written by an LLM from randomly chosen chunks of the most informative sections
(Business, Management's Discussion, Financial Statements). Each question comes with its answer and the chunk it came from.
This is a standard way to build RAG test sets quickly. The questions were then **reviewed by hand**: trivia and
answers too long to grade fairly were removed, and two unclear questions were reworded.

**Hand-written questions** cover what users ask most: each company's headline revenue and net income, with answers
checked in the reports. One is a deliberate trap: Amazon's tables list years oldest first (`2023 | 2024 | 2025`),
so its 2025 net income is the *last* number in the row.

**Out-of-scope questions** are questions the reports can't answer. The correct behavior is to refuse.

The reviewed test set is saved in `eval_set.csv` (the code below only generates questions if that file is missing).

In [3]:
QUESTION_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You write test questions for a chatbot that answers questions about company annual reports.\n"
     "Read the text and write ONE question that can be answered from it, with its answer.\n"
     "Rules:\n"
     "- Name the company ({company}) in the question, and the year or period if the fact depends on it.\n"
     "- Prefer a specific fact: a number, name, place or date.\n"
     "- The answer must be short and written exactly as it appears in the text.\n"
     "- If the text has no clear fact, return {{\"question\": null}}.\n"
     "Reply with JSON only: {{\"question\": \"...\", \"answer\": \"...\"}}"),
    ("human", "{text}"),
])
KEY_SECTIONS = r"^Item (1|7|8|15)\."      # NVIDIA's financial statements are in Item 15


def generate_question(chunk):
    reply = (QUESTION_PROMPT | helper_llm).invoke({"company": COMPANIES[chunk["ticker"]], "text": chunk["text"]})
    try:
        data = json.loads(reply.content)
    except json.JSONDecodeError:
        return None
    return data if data.get("question") and data.get("answer") else None


OUT_OF_SCOPE = [
    "Who is the CEO of Google?",
    "What was Meta's total revenue in 2025?",
    "How many stores does Walmart operate?",
    "What will Apple's revenue be in 2030?",
    "What is the capital of France?",
    "How many subscribers does Netflix have?",
]

EVAL_FILE = EVAL_DIR / "eval_set.csv"
if EVAL_FILE.exists():
    eval_set = pd.read_csv(EVAL_FILE)
else:
    candidates = chunks[chunks["section"].str.match(KEY_SECTIONS) & (chunks["text"].str.split().str.len() > 100)]
    sample = candidates.groupby("ticker").sample(6, random_state=42)
    rows = []
    for chunk in sample.to_dict("records"):
        q = generate_question(chunk)
        if q:
            rows.append({"question": q["question"], "answer": str(q["answer"]), "ticker": chunk["ticker"],
                         "source_chunk": chunk["chunk_id"], "type": "in_scope"})
    rows += [{"question": q, "answer": NOT_FOUND, "ticker": None, "source_chunk": None, "type": "out_of_scope"}
             for q in OUT_OF_SCOPE]
    eval_set = pd.DataFrame(rows)
    eval_set.to_csv(EVAL_FILE, index=False)

print(eval_set.groupby(["type", "origin"]).size().to_string())
eval_set.head(10)

type          origin     
in_scope      generated      16
              handwritten    11
out_of_scope  handwritten     6


,question,answer,ticker,source_chunk,type,origin
0,What is the amount of California R&D credit ca...,$5.4 billion,AAPL,AAPL_186,in_scope,generated
1,What is the date on which Apple Inc.'s fiscal ...,the last Saturday of September,AAPL,AAPL_161,in_scope,generated
2,Which new product did Apple announce in the fi...,14-inch MacBook Pro,AAPL,AAPL_132,in_scope,generated
3,"According to Amazon's annual report, what does...",the portion of payments representing principal...,AMZN,AMZN_133,in_scope,generated
4,"According to Amazon's 10-K, what is Amazon's u...","long-term, sustainable growth in free cash flow",AMZN,AMZN_93,in_scope,generated
5,What was the cash consideration for Amazon's a...,$3.5 billion,AMZN,AMZN_210,in_scope,generated
6,What was the amount of severance costs recorde...,$2.7 billion,AMZN,AMZN_161,in_scope,generated
7,When was Satya Nadella appointed Chairman of t...,June 2021,MSFT,MSFT_48,in_scope,generated
8,What was Microsoft’s revenue from Server produ...,"129,425",MSFT,MSFT_314,in_scope,generated
9,What unified set of cloud-based intelligent bu...,Dynamics 365,MSFT,MSFT_23,in_scope,generated


## Step 2: Which chunks count as "correct" for each question?

A fact often appears in more than one chunk (e.g. revenue is in both the Management's Discussion and the Financial Statements,
and neighboring chunks overlap). So a chunk counts as **relevant** if it's the source chunk **or** it contains the answer text.
Tables write `478,834` where the answer says `$478,834 million`, so the `$` and the word *million* are removed before searching.

In [4]:
def relevant_chunks(row):
    relevant = {row["source_chunk"]}
    answer = re.sub(r"^\$|\s*million$", "", str(row["answer"]).strip(" .")).lower()
    if len(answer) >= 3:
        same_company = chunks[chunks["ticker"] == row["ticker"]]
        relevant |= set(same_company[same_company["text"].str.lower().str.contains(answer, regex=False)]["chunk_id"])
    return relevant


in_scope = eval_set[eval_set["type"] == "in_scope"].copy()
in_scope["relevant"] = in_scope.apply(relevant_chunks, axis=1)
print("Average relevant chunks per question:", round(in_scope["relevant"].apply(len).mean(), 1))

Average relevant chunks per question: 3.4


## Step 3: Evaluate retrieval (no LLM needed, so it's free)

For every question, each search method returns its top 10 chunks. Then:

- **Hit rate@k:** share of questions where a relevant chunk is in the top *k*. If the right chunk isn't retrieved, the LLM can't answer.
- **MRR (mean reciprocal rank):** rewards finding it *early*: 1st place = 1, 2nd = 1/2, 3rd = 1/3, not found = 0.

In [5]:
def first_hit_rank(retrieved, relevant):
    for rank, chunk_id in enumerate(retrieved, start=1):
        if chunk_id in relevant:
            return rank
    return None


retrieval_rows = []
for method in METHODS:
    ranks = [first_hit_rank(retriever.search(q, k=10, method=method), rel)
             for q, rel in zip(in_scope["question"], in_scope["relevant"])]
    row = {"Method": method}
    for k in (3, 5, 10):
        row[f"Hit rate@{k}"] = sum(r is not None and r <= k for r in ranks) / len(ranks)
    row["MRR"] = sum(1 / r for r in ranks if r) / len(ranks)
    retrieval_rows.append(row)

retrieval_results = pd.DataFrame(retrieval_rows).set_index("Method").round(3)
retrieval_results.to_csv(EVAL_DIR / "retrieval_results.csv")
retrieval_results

,Hit rate@3,Hit rate@5,Hit rate@10,MRR
Method,,,,
vector,0.556,0.556,0.704,0.471
vector_filter,0.556,0.556,0.704,0.471
hybrid_rerank,0.741,0.778,0.889,0.685


## Step 4: Evaluate the answers

Every question (in-scope and out-of-scope) is answered by the full chatbot with two search methods:
the plain baseline (`vector`) and the best one (`hybrid_rerank`), both with k=5 chunks.

**Grading:**
- **In-scope:** a second LLM (the *judge*) compares the chatbot's answer with the reference answer.
  Numbers written differently still count (`$4.5 billion` = `4,500 million`).
- **Out-of-scope:** correct only if the chatbot refuses with *"I couldn't find that in the reports."*

Note: the judge is an LLM too (gpt-oss-20b), so grading can occasionally be wrong; a sample of verdicts is shown below to check.

In [6]:
JUDGE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You grade a chatbot's answer against a reference answer.\n"
     "The answer is correct if it states the same fact as the reference. Ignore wording and formatting;\n"
     "numbers in different units or formats count as equal (e.g. 4.5 billion = 4,500 million).\n"
     "If the chatbot says it couldn't find the answer, it is incorrect.\n"
     "Reply with JSON only: {{\"correct\": true}} or {{\"correct\": false}}"),
    ("human", "Question: {question}\nReference answer: {reference}\nChatbot answer: {answer}"),
])


def judge(question, reference, answer):
    reply = (JUDGE_PROMPT | helper_llm).invoke({"question": question, "reference": reference, "answer": answer})
    match = re.search(r"\"correct\"\s*:\s*(true|false)", reply.content.lower())
    return bool(match) and match.group(1) == "true"


def evaluate_answers(method, k=5):
    """Answer every test question with one search method and k chunks. Progress is saved after each
    question, so if the Groq rate limit stops the run, re-running continues where it left off.
    k=5 uses the original files; other values get their own file (e.g. answers_hybrid_rerank_k10.csv)."""
    suffix = "" if k == 5 else f"_k{k}"
    path = EVAL_DIR / f"answers_{method}{suffix}.csv"
    rows = pd.read_csv(path).to_dict("records") if path.exists() else []
    done = {r["question"] for r in rows}
    for row in eval_set.to_dict("records"):
        if row["question"] in done:
            continue
        sources = retriever.get_chunks(retriever.search(row["question"], k=k, method=method))
        answer = generate_answer(answer_llm, row["question"], sources)
        refused = NOT_FOUND.lower().rstrip(".") in answer.lower()
        if row["type"] == "out_of_scope":
            correct = refused
        else:
            correct = (not refused) and judge(row["question"], row["answer"], answer)
        rows.append({**row, "method": method, "k": k, "chatbot_answer": answer, "refused": refused,
                     "correct": correct, "sources": [s["chunk_id"] for s in sources]})
        pd.DataFrame(rows).to_csv(path, index=False)
        print(f"{method:14s} k={k:2d} {len(rows):2d}/{len(eval_set)}  correct={correct}")
    result = pd.DataFrame(rows)
    result["k"] = k
    return result


answers = pd.concat([evaluate_answers(m) for m in ["vector", "hybrid_rerank"]], ignore_index=True)

### Experiment: send 10 chunks instead of 5

Retrieval found a relevant chunk in the top 10 for 89% of questions, but only 78% in the top 5. This run keeps everything else the same (model, prompt, questions) and only changes **k**. Results are saved to `answers_hybrid_rerank_k10.csv`.

In [7]:
answers_k10 = evaluate_answers("hybrid_rerank", k=10)

hybrid_rerank  k=10  1/33  correct=True


hybrid_rerank  k=10  2/33  correct=True


hybrid_rerank  k=10  3/33  correct=True


hybrid_rerank  k=10  4/33  correct=True


hybrid_rerank  k=10  5/33  correct=False


hybrid_rerank  k=10  6/33  correct=True


... (23 outputs left out)

hybrid_rerank  k=10 30/33  correct=True


hybrid_rerank  k=10 31/33  correct=True


hybrid_rerank  k=10 32/33  correct=True


hybrid_rerank  k=10 33/33  correct=True


In [8]:
def summarize(group):
    ins, oos = group[group["type"] == "in_scope"], group[group["type"] == "out_of_scope"]
    return pd.Series({
        "In-scope accuracy": ins["correct"].mean(),
        "Wrongly refused (in-scope)": ins["refused"].mean(),
        "Correctly refused (out-of-scope)": oos["correct"].mean(),
        "Overall accuracy": group["correct"].mean(),
    })


answer_results = answers.groupby("method")[["type", "correct", "refused"]].apply(summarize).round(3)
answer_results.to_csv(EVAL_DIR / "answer_results.csv")
answer_results

,In-scope accuracy,Wrongly refused (in-scope),Correctly refused (out-of-scope),Overall accuracy
method,,,,
hybrid_rerank,0.778,0.185,1.0,0.818
vector,0.556,0.333,1.0,0.636


In [9]:
all_answers = pd.concat([answers, answers_k10], ignore_index=True)
comparison = (all_answers.groupby(["method", "k"])[["type", "correct", "refused"]]
              .apply(summarize).round(3))
comparison.to_csv(EVAL_DIR / "answer_results_by_k.csv")
comparison

In-scope accuracy  Wrongly refused (in-scope)  \
method        k                                                   
hybrid_rerank 5               0.778                       0.185   
              10              0.852                       0.111   
vector        5               0.556                       0.333   

                  Correctly refused (out-of-scope)  Overall accuracy  
method        k                                                       
hybrid_rerank 5                                1.0             0.818  
              10                               1.0             0.879  
vector        5                                1.0             0.636

In [10]:
k5 = answers[answers["method"] == "hybrid_rerank"].set_index("question")["correct"]
k10 = answers_k10.set_index("question")["correct"]
changes = pd.DataFrame({"k5": k5, "k10": k10.reindex(k5.index)})
print("Fixed by k=10:")
display(answers_k10.set_index("question").loc[changes[~changes["k5"] & changes["k10"]].index, ["answer", "chatbot_answer"]])
print("Broken by k=10:")
display(answers_k10.set_index("question").loc[changes[changes["k5"] & ~changes["k10"]].index, ["answer", "chatbot_answer"]])

Fixed by k=10:


,answer,chatbot_answer
question,,
When was NVIDIA's Compensation Recovery Policy amended and restated?,"November 30, 2023",NVIDIA’s Compensation Recovery Policy was amen...
What was Microsoft's total revenue in fiscal year 2026?,"$331,839 million",Microsoft’s total revenue for fiscal year 2026...


Broken by k=10:


,answer,chatbot_answer
question,,


### Result: 10 chunks is better

| `hybrid_rerank` | k=5 | **k=10** |
|---|---|---|
| In-scope accuracy (27) | 0.778 | **0.852** |
| Wrongly refused (in-scope) | 0.185 | **0.111** |
| Correctly refused (6 out-of-scope) | 1.000 | **1.000** |
| **Overall accuracy (33)** | 0.818 | **0.879** |

- **2 questions fixed, 0 broken.** One fix is Microsoft's total revenue, a known failure at k=5: the income-statement chunk
  was ranked just below the top 5.
- The gain comes from fewer wrong refusals: the answer was retrieved but cut off before reaching the LLM.
- More context did **not** make the chatbot invent answers: all out-of-scope questions are still refused.
- Cost: about twice the input tokens per answer. Worth it here, so **k=10 is now the default** in the pipeline and the app.
- Still wrong: Microsoft's net income and Amazon's employee count (refused), one NVIDIA tax question (a different tax line
  was reported), and one vague Amazon question.

## Step 5: Look at examples

In [11]:
pd.set_option("display.max_colwidth", 120)
best = answers[(answers["method"] == "hybrid_rerank") & (answers["type"] == "in_scope")]
best[["question", "answer", "chatbot_answer", "correct"]].head(8)

,question,answer,chatbot_answer,correct
33,"What is the amount of California R&D credit carryforwards that Apple had as of September 26, 2026?",$5.4 billion,"Apple had **$5.4 billion** in California R&D credit carryforwards as of September 26, 2026[AAPL_186].",True
34,What is the date on which Apple Inc.'s fiscal year ends for the 2026 annual reporting period?,the last Saturday of September,Apple Inc.’s fiscal year ends on the last Saturday of September; for the 2026 annual reporting period it ended on **...,True
35,Which new product did Apple announce in the first quarter of fiscal year 2026?,14-inch MacBook Pro,Apple announced the 14‑inch MacBook Pro in the first quarter of fiscal 2026[AAPL_132].,True
36,"According to Amazon's annual report, what does free cash flow not incorporate?",the portion of payments representing principal reductions of debt or cash payments for business acquisitions.,Free cash flow does not incorporate the portion of payments representing principal reductions of debt or cash paymen...,True
37,"According to Amazon's 10-K, what is Amazon's ultimate financial focus?","long-term, sustainable growth in free cash flow",I couldn't find that in the reports.,False
38,"What was the cash consideration for Amazon's acquisition of 1Life Healthcare, Inc. in 2023?",$3.5 billion,"The cash consideration for Amazon’s acquisition of 1Life Healthcare, Inc. (One Medical) in 2023 was approximately **...",True
39,"What was the amount of severance costs recorded by Amazon for the year ended December 31, 2025?",$2.7 billion,"Amazon recorded approximately **$2.7 billion** of severance costs for the year ended December 31, 2025[AMZN_161].",True
40,When was Satya Nadella appointed Chairman of the Board at Microsoft?,June 2021,Satya Nadella was appointed Chairman of the Board in June 2021[MSFT_48].,True


In [12]:
# Questions the best method still gets wrong
best[~best["correct"]][["question", "answer", "chatbot_answer"]]

,question,answer,chatbot_answer
37,"According to Amazon's 10-K, what is Amazon's ultimate financial focus?","long-term, sustainable growth in free cash flow",I couldn't find that in the reports.
43,When was NVIDIA's Compensation Recovery Policy amended and restated?,"November 30, 2023",I couldn't find that in the reports.
46,"What was NVIDIA's federal tax expense for the year ended January 26, 2025?","17,645","NVIDIA’s federal income‑tax expense for the fiscal year ended January 26 2025 was **$14,032 million**[NVDA_294]."
51,What was Microsoft's total revenue in fiscal year 2026?,"$331,839 million",I couldn't find that in the reports.
52,What was Microsoft's net income in fiscal year 2026?,"$133,749 million",I couldn't find that in the reports.
59,How many employees did Amazon have at the end of 2025?,"approximately 1,576,000",I couldn't find that in the reports.


In [13]:
# Questions fixed by hybrid search + reranker
baseline = answers[(answers["method"] == "vector")].set_index("question")["correct"]
improved = answers[(answers["method"] == "hybrid_rerank")].set_index("question")["correct"]
fixed = improved[improved & ~baseline.reindex(improved.index).fillna(False).astype(bool)].index
answers[(answers["method"] == "hybrid_rerank") & answers["question"].isin(fixed)][["question", "chatbot_answer"]]

,question,chatbot_answer
33,"What is the amount of California R&D credit carryforwards that Apple had as of September 26, 2026?","Apple had **$5.4 billion** in California R&D credit carryforwards as of September 26, 2026[AAPL_186]."
36,"According to Amazon's annual report, what does free cash flow not incorporate?",Free cash flow does not incorporate the portion of payments representing principal reductions of debt or cash paymen...
48,"According to Tesla's 10-K, what is every Tesla vehicle delivered today designed for?",Every Tesla vehicle delivered today is designed for autonomy[TSLA_35].
50,What was Apple's net income in fiscal 2026?,"Apple reported net income of **$132,175 million** for fiscal year 2026[AAPL_154]."
53,What was NVIDIA's total revenue in fiscal year 2026?,"NVIDIA reported total revenue of **$215,938 million** for fiscal year 2026[NVDA_230]."
54,What was NVIDIA's net income in fiscal year 2026?,"NVIDIA reported net income of **$120,067 million** for fiscal year 2026[NVDA_230]."
55,What were Tesla's total revenues in 2025?,"Tesla reported total revenues of **$94.83 billion** for the year ended December 31, 2025[TSLA_167]."
56,What was Tesla's net income attributable to common stockholders in 2025?,"Tesla’s net income attributable to common stockholders in 2025 was **$3,794 million**[TSLA_281]."


## Conclusion

| | Baseline (`vector`) | **Best (`hybrid_rerank`)** |
|---|---|---|
| Retrieval: hit rate@5 | 0.556 | **0.778** |
| Retrieval: MRR | 0.471 | **0.685** |
| In-scope accuracy (27 questions) | 0.556 | **0.778** |
| Wrongly refused (in-scope) | 0.333 | 0.185 |
| Correctly refused (6 out-of-scope) | 1.000 | **1.000** |
| **Overall accuracy (33 questions)** | 0.636 | **0.818** |

**Update:** sending 10 chunks instead of 5 raised the best method's overall accuracy to **0.879** (see the experiment above).

**What the numbers show**
- **Hybrid search + reranker is the biggest improvement:** +22 points of answer accuracy. Better retrieval directly means better answers.
- **The company filter changes nothing** (same scores as plain `vector`): embedding search already finds the right company.
  The hard part is finding the right *chunk* inside that company's report.
- **The chatbot never answered an out-of-scope question.** It refused all 6.
- **Most failures are safe:** when the right chunk isn't retrieved, the chatbot says it couldn't find the answer instead of guessing.
  Only 1 of 27 in-scope answers gave a wrong number (NVIDIA's federal tax expense).
- **The year-order trap passed:** Amazon's 2025 net income was read correctly from a table that lists years oldest first.

**Remaining failures:** Microsoft's headline revenue and net income (retrieval misses the income-statement chunk),
Amazon's employee count, and two questions answered in prose rather than tables.

**Limits of this evaluation:** 33 questions is a small test set (each question is ~3 points of accuracy), most in-scope questions
were generated by an LLM from the documents, and the judge is an LLM.

**Next steps:** a larger test set, more chunks for the LLM (hit rate@10 is 0.889), and table-aware chunking that keeps each table's header with its rows.